# 08 — MySQL Database Update

## Objective
Inspect the existing MySQL database and validate the four processed CSV datasets before making any database changes.

## Safety
This notebook performs read-only inspection initially.
No database modifications will be performed until the inspection is reviewed.

In [274]:
from pathlib import Path
import pandas as pd
import sqlalchemy
from sqlalchemy import inspect, text

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Pandas version:", pd.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)

Project root: d:\Git\Personal\ipl-player-performance-v2
Project root exists: True
Pandas version: 3.1.0
SQLAlchemy version: 2.1.6


In [275]:
env_path = PROJECT_ROOT / ".env"

print("Environment file exists:", env_path.exists())
print("Environment file path:", env_path)

Environment file exists: True
Environment file path: d:\Git\Personal\ipl-player-performance-v2\.env


In [276]:
from dotenv import dotenv_values

env_values = dotenv_values(PROJECT_ROOT / ".env")

print("Environment variable names:")
for key in env_values.keys():
    print("-", key)

Environment variable names:
- DJANGO_SECRET_KEY
- DJANGO_DEBUG
- DB_NAME
- DB_USER
- DB_PASSWORD
- DB_HOST
- DB_PORT


In [277]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Load existing project credentials
load_dotenv(PROJECT_ROOT / ".env", override=False)

# Read database configuration
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

# Validate required variables
required_vars = {
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT
}

missing_vars = [key for key, value in required_vars.items() if not value]

if missing_vars:
    raise ValueError(f"Missing database variables: {missing_vars}")

# Build SQLAlchemy connection URL safely
database_url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(
    database_url,
    pool_pre_ping=True,
    echo=False
)

print("Database configuration loaded.")
print("Database name:", DB_NAME)
print("Database host:", DB_HOST)
print("SQLAlchemy engine created.")

Database configuration loaded.
Database name: ipl_analytics
Database host: 127.0.0.1
SQLAlchemy engine created.


## 2. MySQL Connection Verification

Verify the existing connection to the `ipl_analytics` database.
Only read-only queries will be executed.

In [278]:
with engine.connect() as connection:
    database_info = connection.execute(
        text("SELECT DATABASE(), VERSION()")
    ).fetchone()

print("Database:", database_info[0])
print("MySQL Version:", database_info[1])

assert database_info[0] == "ipl_analytics", "Incorrect database connected!"
print("MYSQL CONNECTION: PASS")

Database: ipl_analytics
MySQL Version: 8.0.46
MYSQL CONNECTION: PASS


## 3. Existing MySQL Database Inspection

Inspect existing tables, their columns, primary keys, foreign keys and row counts.

This step is strictly read-only.

In [279]:
from sqlalchemy import inspect, text

inspector = inspect(engine)

existing_tables = inspector.get_table_names()

print("DATABASE:", DB_NAME)
print("\nEXISTING TABLES:")
for table in existing_tables:
    print("-", table)

print("\nTABLE ROW COUNTS:")

table_counts = {}

with engine.connect() as connection:
    for table in existing_tables:
        count = connection.execute(
            text(f"SELECT COUNT(*) FROM `{table}`")
        ).scalar_one()

        table_counts[table] = count
        print(f"{table}: {count:,} rows")

DATABASE: ipl_analytics

EXISTING TABLES:
- batting_next_match_features
- batting_player_match_regular
- bowling_next_match_features
- bowling_player_match_regular
- deliveries
- innings
- matches

TABLE ROW COUNTS:
batting_next_match_features: 18,822 rows
batting_player_match_regular: 18,822 rows
bowling_next_match_features: 14,700 rows
bowling_player_match_regular: 14,700 rows
deliveries: 295,732 rows
innings: 2,514 rows
matches: 1,243 rows


In [280]:
schema_details = []

for table in existing_tables:
    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print("=" * 70)

    columns = inspector.get_columns(table)
    primary_keys = inspector.get_pk_constraint(table)
    foreign_keys = inspector.get_foreign_keys(table)

    for column in columns:
        print(
            f"{column['name']:<35} "
            f"{str(column['type']):<25} "
            f"Nullable={column['nullable']}"
        )

        schema_details.append({
            "table": table,
            "column": column["name"],
            "sql_type": str(column["type"]),
            "nullable": column["nullable"]
        })

    print("\nPrimary Key:", primary_keys.get("constrained_columns"))
    print("Foreign Keys:", foreign_keys)

existing_schema_df = pd.DataFrame(schema_details)

print("\nTOTAL EXISTING COLUMNS:", len(existing_schema_df))


TABLE: batting_next_match_features
match_id                            VARCHAR(20) COLLATE utf8mb4_unicode_ci Nullable=False
player                              VARCHAR(100) COLLATE utf8mb4_unicode_ci Nullable=False
is_super_over                       TINYINT                   Nullable=False
batting_team                        VARCHAR(100) COLLATE utf8mb4_unicode_ci Nullable=False
runs_scored                         INTEGER                   Nullable=False
balls_faced                         INTEGER                   Nullable=False
fours                               INTEGER                   Nullable=False
sixes                               INTEGER                   Nullable=False
dismissals                          INTEGER                   Nullable=False
retired_hurt_count                  INTEGER                   Nullable=False
strike_rate                         DOUBLE                    Nullable=True
batting_average                     DOUBLE                    Nullable=True
i

## 4. Player-Match and Feature CSV Inspection

Inspect the four processed datasets before designing new MySQL tables.

Validate:
- File existence
- Dataset dimensions
- Column names and data types
- Missing values
- Duplicate rows
- Candidate primary keys

In [281]:
from pathlib import Path
import pandas as pd

PLAYER_MATCH_DIR = PROJECT_ROOT / "data" / "processed" / "player_match"

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"

csv_files = {
    "batting_player_match_regular": (
        PLAYER_MATCH_DIR / "batting_player_match_regular.csv"
    ),
    "bowling_player_match_regular": (
        PLAYER_MATCH_DIR / "bowling_player_match_regular.csv"
    ),
    "batting_next_match_features": (
        FEATURE_DIR / "batting_next_match_features.csv"
    ),
    "bowling_next_match_features": (
        FEATURE_DIR / "bowling_next_match_features.csv"
    )
}

csv_data = {}

for name, file_path in csv_files.items():

    print("\n" + "=" * 75)
    print("DATASET:", name)
    print("=" * 75)

    print("File:", file_path)
    print("Exists:", file_path.exists())

    if not file_path.exists():
        raise FileNotFoundError(f"Missing dataset: {file_path}")

    df = pd.read_csv(file_path)
    csv_data[name] = df

    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("\nColumns and data types:")
    print(df.dtypes.to_string())

    print("\nMissing values:")
    print(df.isna().sum().to_string())

print("\nALL FOUR CSV FILES LOADED SUCCESSFULLY")


DATASET: batting_player_match_regular
File: d:\Git\Personal\ipl-player-performance-v2\data\processed\player_match\batting_player_match_regular.csv
Exists: True
Shape: (18822, 17)
Duplicate rows: 0

Columns and data types:
match_id                int64
player                    str
is_super_over            bool
batting_team              str
runs_scored             int64
balls_faced             int64
fours                   int64
sixes                   int64
dismissals              int64
retired_hurt_count      int64
strike_rate           float64
batting_average       float64
innings_batted          int64
not_out_innings         int64
season                    str
date                      str
match_type                str

Missing values:
match_id                 0
player                   0
is_super_over            0
batting_team             0
runs_scored              0
balls_faced              0
fours                    0
sixes                    0
dismissals               0
retired

Duplicate rows: 0

Columns and data types:
match_id                             int64
player                                 str
is_super_over                         bool
runs_conceded                        int64
legal_balls                          int64
wides_conceded                       int64
noballs_conceded                     int64
innings_bowled                       int64
wickets                              int64
overs_bowled                       float64
economy_rate                       float64
bowling_average                    float64
bowling_strike_rate                float64
season                                 str
date                                   str
match_type                             str
target_next_match_wickets          float64
target_next_match_runs_conceded    float64
target_next_match_legal_balls      float64
target_next_match_economy_rate     float64
previous_match_date                    str
days_since_previous_match          float64
previous_wi

## 5. Primary Key and Foreign Key Compatibility Audit

Validate:
- Uniqueness of (match_id, player)
- Match ID compatibility with MySQL
- Missing identifiers
- Date parsing
- Super-over records
- Match references against existing database

In [282]:
# Candidate primary-key validation
print("=" * 75)
print("CANDIDATE PRIMARY KEY VALIDATION")
print("=" * 75)

for name, df in csv_data.items():

    duplicate_keys = df.duplicated(
        subset=["match_id", "player"]
    ).sum()

    print(f"\n{name}")
    print("Rows:", len(df))
    print("Duplicate (match_id, player) keys:", duplicate_keys)
    print("Missing match_id:", df["match_id"].isna().sum())
    print("Missing player:", df["player"].isna().sum())


# Match ID compatibility audit
with engine.connect() as connection:

    mysql_match_ids = pd.read_sql(
        text("SELECT match_id FROM matches"),
        connection
    )

mysql_match_ids["match_id_normalized"] = (
    mysql_match_ids["match_id"].astype(str).str.strip()
)

print("\n" + "=" * 75)
print("MATCH ID COMPATIBILITY")
print("=" * 75)

for name in [
    "batting_player_match_regular",
    "bowling_player_match_regular"
]:

    csv_ids = set(
        csv_data[name]["match_id"].astype(str).str.strip()
    )

    db_ids = set(mysql_match_ids["match_id_normalized"])

    unmatched_ids = csv_ids - db_ids

    print(f"\n{name}")
    print("Unique CSV match IDs:", len(csv_ids))
    print("Unmatched IDs:", len(unmatched_ids))
    print("Sample unmatched IDs:", list(unmatched_ids)[:10])


# Date validation
print("\n" + "=" * 75)
print("DATE VALIDATION")
print("=" * 75)

for name, df in csv_data.items():

    parsed_dates = pd.to_datetime(df["date"], errors="coerce")

    print(
        name,
        "| Invalid dates:",
        parsed_dates.isna().sum()
    )


# Super-over distribution
print("\n" + "=" * 75)
print("SUPER-OVER DISTRIBUTION")
print("=" * 75)

for name, df in csv_data.items():

    print(f"\n{name}")
    print(df["is_super_over"].value_counts(dropna=False))

CANDIDATE PRIMARY KEY VALIDATION

batting_player_match_regular
Rows: 18822
Duplicate (match_id, player) keys: 0
Missing match_id: 0
Missing player: 0

bowling_player_match_regular
Rows: 14700
Duplicate (match_id, player) keys: 0
Missing match_id: 0
Missing player: 0

batting_next_match_features
Rows: 18822
Duplicate (match_id, player) keys: 0
Missing match_id: 0
Missing player: 0

bowling_next_match_features
Rows: 14700
Duplicate (match_id, player) keys: 0
Missing match_id: 0
Missing player: 0

MATCH ID COMPATIBILITY

batting_player_match_regular
Unique CSV match IDs: 1243
Unmatched IDs: 0
Sample unmatched IDs: []

bowling_player_match_regular
Unique CSV match IDs: 1243
Unmatched IDs: 0
Sample unmatched IDs: []

DATE VALIDATION
batting_player_match_regular | Invalid dates: 0
bowling_player_match_regular | Invalid dates: 0
batting_next_match_features | Invalid dates: 0
bowling_next_match_features | Invalid dates: 0

SUPER-OVER DISTRIBUTION

batting_player_match_regular
is_super_over
Fal

In [283]:
print("=" * 85)
print("CSV TO MYSQL DATA TYPE MAPPING")
print("=" * 85)

for name, df in csv_data.items():

    print(f"\n{'=' * 85}")
    print("DATASET:", name)
    print("=" * 85)

    for column in df.columns:

        dtype = str(df[column].dtype)

        if column == "match_id":
            mysql_type = "VARCHAR(20)"
        elif column == "player" or column == "batting_team":
            mysql_type = "VARCHAR(100)"
        elif column in ["season", "match_type"]:
            mysql_type = "VARCHAR(30)"
        elif column in ["date", "previous_match_date"]:
            mysql_type = "DATE"
        elif dtype == "bool":
            mysql_type = "BOOLEAN"
        elif dtype.startswith("int"):
            mysql_type = "BIGINT"
        elif dtype.startswith("float"):
            mysql_type = "DOUBLE"
        else:
            mysql_type = "VARCHAR(255)"

        nullable = df[column].isna().any()

        print(
            f"{column:<40} "
            f"{mysql_type:<15} "
            f"Nullable={nullable}"
        )

CSV TO MYSQL DATA TYPE MAPPING

DATASET: batting_player_match_regular
match_id                                 VARCHAR(20)     Nullable=False
player                                   VARCHAR(100)    Nullable=False
is_super_over                            BOOLEAN         Nullable=False
batting_team                             VARCHAR(100)    Nullable=False
runs_scored                              BIGINT          Nullable=False
balls_faced                              BIGINT          Nullable=False
fours                                    BIGINT          Nullable=False
sixes                                    BIGINT          Nullable=False
dismissals                               BIGINT          Nullable=False
retired_hurt_count                       BIGINT          Nullable=False
strike_rate                              DOUBLE          Nullable=True
batting_average                          DOUBLE          Nullable=True
innings_batted                           BIGINT          Nullable=Fa

## 6. Refined SQL Data Type Validation

Identify floating-point columns containing only whole-number values.

This helps distinguish:
- Integer-valued targets and historical counts
- Continuous metrics and rolling averages

No database changes will be performed.

In [284]:
print("=" * 85)
print("FLOAT COLUMN INTEGER-VALUE AUDIT")
print("=" * 85)

for name, df in csv_data.items():

    print(f"\nDATASET: {name}")

    float_columns = df.select_dtypes(
        include=["float"]
    ).columns

    for column in float_columns:

        values = df[column].dropna()

        if values.empty:
            continue

        is_integer_valued = (
            values.mod(1).abs() < 1e-9
        ).all()

        print(
            f"{column:<42} "
            f"Integer-valued={is_integer_valued} "
            f"Non-null={len(values)}"
        )

FLOAT COLUMN INTEGER-VALUE AUDIT

DATASET: batting_player_match_regular
strike_rate                                Integer-valued=False Non-null=18763
batting_average                            Integer-valued=True Non-null=14655

DATASET: bowling_player_match_regular
overs_bowled                               Integer-valued=False Non-null=14700
economy_rate                               Integer-valued=False Non-null=14699
bowling_average                            Integer-valued=False Non-null=8449
bowling_strike_rate                        Integer-valued=False Non-null=8449

DATASET: batting_next_match_features
strike_rate                                Integer-valued=False Non-null=18763
batting_average                            Integer-valued=True Non-null=14655
target_next_match_runs_scored              Integer-valued=True Non-null=18083
target_next_match_balls_faced              Integer-valued=True Non-null=18083
target_next_match_fours                    Integer-valued=True Non-

## 7. Proposed MySQL Table Definitions

Generate CREATE TABLE statements for four datasets.

Requirements:
- Composite primary key (match_id, player)
- Foreign key to matches(match_id)
- InnoDB engine
- utf8mb4 encoding
- Preserve NULL values
- No database execution during this stage

In [285]:
from sqlalchemy import MetaData, Table

# Reuse the existing metadata object from the previous cell.
# Do NOT create a new MetaData() object.

matches_table = Table(
    "matches",
    metadata,
    autoload_with=engine
)

print("Reflected table:", matches_table.name)
print("Columns:", list(matches_table.columns.keys()))

print("Primary key:", [
    column.name
    for column in matches_table.primary_key.columns
])

assert "match_id" in matches_table.columns

print("MATCHES METADATA REFLECTION: PASS")

Reflected table: matches
Columns: ['match_id', 'source_file', 'season', 'date', 'match_type', 'gender', 'team_type', 'overs_limit', 'venue', 'city', 'city_original', 'city_was_imputed', 'team1', 'team2', 'teams_json', 'toss_winner', 'toss_decision', 'winner', 'result', 'method', 'eliminator', 'bowl_out', 'player_of_match', 'event_name', 'event_match_number', 'event_stage']
Primary key: ['match_id']
MATCHES METADATA REFLECTION: PASS


In [286]:
from sqlalchemy import (
    Integer, Boolean, String, Date, Double,
    Table, Column, ForeignKeyConstraint,
    PrimaryKeyConstraint
)
from sqlalchemy.schema import CreateTable

# Remove previously generated tables from metadata.
# Preserve the reflected existing matches table.
for name in csv_data.keys():
    if name in metadata.tables:
        metadata.remove(metadata.tables[name])

# Integer-valued columns
integer_columns = {
    "runs_scored", "balls_faced", "fours", "sixes",
    "dismissals", "retired_hurt_count", "innings_batted",
    "not_out_innings", "runs_conceded", "legal_balls",
    "wides_conceded", "noballs_conceded", "innings_bowled",
    "wickets", "matches_played_before"
}

integer_target_columns = {
    "target_next_match_runs_scored",
    "target_next_match_balls_faced",
    "target_next_match_fours",
    "target_next_match_sixes",
    "target_next_match_dismissals",
    "target_next_match_wickets",
    "target_next_match_runs_conceded",
    "target_next_match_legal_balls",
    "days_since_previous_match"
}

integer_history_columns = {
    "previous_runs_scored",
    "previous_balls_faced",
    "previous_fours",
    "previous_sixes",
    "previous_dismissals",
    "previous_wickets",
    "previous_runs_conceded",
    "previous_legal_balls"
}


def get_sql_type(column):

    if column == "match_id":
        return String(20)

    if column == "player":
        return String(100)

    if column == "season":
        return String(30)

    if column == "match_type":
        return String(30)

    if column == "batting_team":
        return String(100)

    if column in {"date", "previous_match_date"}:
        return Date()

    if column == "is_super_over":
        return Boolean()

    if (
        column in integer_columns
        or column in integer_target_columns
        or column in integer_history_columns
    ):
        return Integer()

    # Rates, averages, rolling means and other decimal metrics
    return Double()


# Generate four proposed table definitions
proposed_tables = {}

for name, df in csv_data.items():

    columns = []

    for column in df.columns:

        nullable = bool(df[column].isna().any())

        columns.append(
            Column(
                column,
                get_sql_type(column),
                nullable=nullable
            )
        )

    table = Table(
        name,
        metadata,
        *columns,
        PrimaryKeyConstraint("match_id", "player"),
        ForeignKeyConstraint(
            ["match_id"],
            ["matches.match_id"],
            name=f"fk_{name}_matches"
        ),
        mysql_engine="InnoDB",
        mysql_charset="utf8mb4"
    )

    proposed_tables[name] = table

    print("\n" + "=" * 85)
    print("DDL PREVIEW:", name)
    print("=" * 85)

    print(
        CreateTable(table).compile(
            dialect=engine.dialect
        )
    )

print("\nDDL GENERATION COMPLETED — PREVIEW ONLY")


DDL PREVIEW: batting_player_match_regular

CREATE TABLE batting_player_match_regular (
	match_id VARCHAR(20) NOT NULL, 
	player VARCHAR(100) NOT NULL, 
	is_super_over BOOL NOT NULL, 
	batting_team VARCHAR(100) NOT NULL, 
	runs_scored INTEGER NOT NULL, 
	balls_faced INTEGER NOT NULL, 
	fours INTEGER NOT NULL, 
	sixes INTEGER NOT NULL, 
	dismissals INTEGER NOT NULL, 
	retired_hurt_count INTEGER NOT NULL, 
	strike_rate DOUBLE, 
	batting_average DOUBLE, 
	innings_batted INTEGER NOT NULL, 
	not_out_innings INTEGER NOT NULL, 
	season VARCHAR(30) NOT NULL, 
	date DATE NOT NULL, 
	match_type VARCHAR(30) NOT NULL, 
	PRIMARY KEY (match_id, player), 
	CONSTRAINT fk_batting_player_match_regular_matches FOREIGN KEY(match_id) REFERENCES matches (match_id)
)ENGINE=InnoDB CHARSET=utf8mb4



DDL PREVIEW: bowling_player_match_regular

CREATE TABLE bowling_player_match_regular (
	match_id VARCHAR(20) NOT NULL, 
	player VARCHAR(100) NOT NULL, 
	is_super_over BOOL NOT NULL, 
	runs_conceded INTEGER NOT NUL

In [287]:
print("=" * 75)
print("FINAL DDL STRUCTURE VALIDATION")
print("=" * 75)

expected_columns = {
    "batting_player_match_regular": 17,
    "bowling_player_match_regular": 16,
    "batting_next_match_features": 56,
    "bowling_next_match_features": 43
}

for name, table in proposed_tables.items():

    print(f"\nTABLE: {name}")

    actual_columns = len(table.columns)
    expected = expected_columns[name]

    primary_key = [
        col.name for col in table.primary_key.columns
    ]

    foreign_keys = [
        fk.target_fullname
        for fk in table.foreign_keys
    ]

    print("Columns:", actual_columns)
    print("Expected columns:", expected)
    print("Primary key:", primary_key)
    print("Foreign keys:", foreign_keys)
    print("Engine:", table.dialect_options["mysql"]["engine"])
    print("Charset:", table.dialect_options["mysql"]["charset"])

    assert actual_columns == expected
    assert primary_key == ["match_id", "player"]
    assert "matches.match_id" in foreign_keys
    assert table.dialect_options["mysql"]["engine"] == "InnoDB"

print("\nFINAL DDL VALIDATION: PASS")

FINAL DDL STRUCTURE VALIDATION

TABLE: batting_player_match_regular
Columns: 17
Expected columns: 17
Primary key: ['match_id', 'player']
Foreign keys: ['matches.match_id']
Engine: InnoDB
Charset: utf8mb4

TABLE: bowling_player_match_regular
Columns: 16
Expected columns: 16
Primary key: ['match_id', 'player']
Foreign keys: ['matches.match_id']
Engine: InnoDB
Charset: utf8mb4

TABLE: batting_next_match_features
Columns: 56
Expected columns: 56
Primary key: ['match_id', 'player']
Foreign keys: ['matches.match_id']
Engine: InnoDB
Charset: utf8mb4

TABLE: bowling_next_match_features
Columns: 43
Expected columns: 43
Primary key: ['match_id', 'player']
Foreign keys: ['matches.match_id']
Engine: InnoDB
Charset: utf8mb4

FINAL DDL VALIDATION: PASS


## 8. MySQL Table Creation Safety Audit

Verify that the four proposed table names do not already exist.

Check existing database state before executing DDL.

No database modifications during this checkpoint.

In [288]:
proposed_names = {
    "batting_player_match_regular",
    "bowling_player_match_regular",
    "batting_next_match_features",
    "bowling_next_match_features"
}

existing_names = set(inspect(engine).get_table_names())

already_exist = proposed_names.intersection(existing_names)
not_created = proposed_names.difference(existing_names)

print("Proposed tables:", sorted(proposed_names))
print("Already existing:", sorted(already_exist))
print("Available for creation:", sorted(not_created))

if already_exist:
    print("\nExisting tables detected.")
    print("Skipping creation to protect existing data.")

if not_created:
    print("\nTables available for creation:", sorted(not_created))
else:
    print("\nAll proposed tables already exist.")

Proposed tables: ['batting_next_match_features', 'batting_player_match_regular', 'bowling_next_match_features', 'bowling_player_match_regular']
Already existing: ['batting_next_match_features', 'batting_player_match_regular', 'bowling_next_match_features', 'bowling_player_match_regular']
Available for creation: []

Existing tables detected.
Skipping creation to protect existing data.

All proposed tables already exist.


In [289]:
from sqlalchemy import text, inspect

print("=" * 75)
print("MYSQL CREATE TABLE ERROR DIAGNOSTICS")
print("=" * 75)

# 1. Check database
with engine.connect() as conn:
    print(
        "Database:",
        conn.execute(text("SELECT DATABASE()")).scalar_one()
    )

# 2. Inspect parent match_id definition
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            COLUMN_NAME,
            COLUMN_TYPE,
            IS_NULLABLE,
            CHARACTER_SET_NAME,
            COLLATION_NAME
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'matches'
          AND COLUMN_NAME = 'match_id'
    """))

    print("\nPARENT COLUMN: matches.match_id")

    for row in result:
        print(dict(row._mapping))

# 3. Inspect storage engines
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT TABLE_NAME, ENGINE
        FROM information_schema.TABLES
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME IN (
              'matches',
              'batting_player_match_regular',
              'bowling_player_match_regular',
              'batting_next_match_features',
              'bowling_next_match_features'
          )
        ORDER BY TABLE_NAME
    """))

    print("\nEXISTING TABLES AND ENGINES")

    for row in result:
        print(dict(row._mapping))

# 4. Check which proposed tables exist
inspector = inspect(engine)

print("\nPROPOSED TABLE STATUS")

for name in proposed_tables:
    print(
        name,
        "EXISTS" if inspector.has_table(name) else "NOT CREATED"
    )

print("\nDIAGNOSTIC COMPLETED")

MYSQL CREATE TABLE ERROR DIAGNOSTICS
Database: ipl_analytics

PARENT COLUMN: matches.match_id
{'COLUMN_NAME': 'match_id', 'COLUMN_TYPE': 'varchar(20)', 'IS_NULLABLE': 'NO', 'CHARACTER_SET_NAME': 'utf8mb4', 'COLLATION_NAME': 'utf8mb4_unicode_ci'}

EXISTING TABLES AND ENGINES
{'TABLE_NAME': 'batting_next_match_features', 'ENGINE': 'InnoDB'}
{'TABLE_NAME': 'batting_player_match_regular', 'ENGINE': 'InnoDB'}
{'TABLE_NAME': 'bowling_next_match_features', 'ENGINE': 'InnoDB'}
{'TABLE_NAME': 'bowling_player_match_regular', 'ENGINE': 'InnoDB'}
{'TABLE_NAME': 'matches', 'ENGINE': 'InnoDB'}

PROPOSED TABLE STATUS
batting_player_match_regular EXISTS
bowling_player_match_regular EXISTS
batting_next_match_features EXISTS
bowling_next_match_features EXISTS

DIAGNOSTIC COMPLETED


In [290]:
from sqlalchemy import text

print("=" * 75)
print("DATABASE DEFAULT COLLATION")
print("=" * 75)

with engine.connect() as conn:

    result = conn.execute(text("""
        SELECT
            @@character_set_database AS db_charset,
            @@collation_database AS db_collation
    """))

    print(dict(result.fetchone()._mapping))


print("\nPROPOSED CHILD COLUMN DEFINITIONS")

for name in proposed_tables:

    column = proposed_tables[name].c.match_id

    print(
        name,
        "| Type:", column.type,
        "| Nullable:", column.nullable
    )

DATABASE DEFAULT COLLATION
{'db_charset': 'utf8mb4', 'db_collation': 'utf8mb4_unicode_ci'}

PROPOSED CHILD COLUMN DEFINITIONS
batting_player_match_regular | Type: VARCHAR(20) | Nullable: False
bowling_player_match_regular | Type: VARCHAR(20) | Nullable: False
batting_next_match_features | Type: VARCHAR(20) | Nullable: False
bowling_next_match_features | Type: VARCHAR(20) | Nullable: False


In [291]:
with engine.connect() as conn:

    result = conn.execute(text("""
        SHOW TABLE STATUS
        WHERE Name = 'matches'
    """))

    row = result.fetchone()

    print("Parent table collation:", row.Collation)

Parent table collation: utf8mb4_unicode_ci


In [292]:
from sqlalchemy import text

with engine.connect() as conn:

    result = conn.execute(
        text("SHOW ENGINE INNODB STATUS")
    ).mappings().first()

    status = result["Status"]

    marker = "LATEST FOREIGN KEY ERROR"

    if marker in status:
        error_section = status.split(marker, 1)[1]

        print("=" * 75)
        print("LATEST INNODB FOREIGN KEY ERROR")
        print("=" * 75)

        print(error_section[:5000])
    else:
        print("No latest foreign-key error section found.")

No latest foreign-key error section found.


In [293]:
with engine.connect() as conn:

    result = conn.execute(
        text("SHOW CREATE TABLE matches")
    ).fetchone()

    print(result[1])

CREATE TABLE `matches` (
  `match_id` varchar(20) COLLATE utf8mb4_unicode_ci NOT NULL,
  `source_file` varchar(100) COLLATE utf8mb4_unicode_ci NOT NULL,
  `season` varchar(10) COLLATE utf8mb4_unicode_ci NOT NULL,
  `date` date NOT NULL,
  `match_type` varchar(20) COLLATE utf8mb4_unicode_ci NOT NULL,
  `gender` varchar(20) COLLATE utf8mb4_unicode_ci NOT NULL,
  `team_type` varchar(30) COLLATE utf8mb4_unicode_ci NOT NULL,
  `overs_limit` smallint NOT NULL,
  `venue` varchar(150) COLLATE utf8mb4_unicode_ci NOT NULL,
  `city` varchar(100) COLLATE utf8mb4_unicode_ci NOT NULL,
  `city_original` varchar(100) COLLATE utf8mb4_unicode_ci DEFAULT NULL,
  `city_was_imputed` tinyint(1) NOT NULL,
  `team1` varchar(100) COLLATE utf8mb4_unicode_ci NOT NULL,
  `team2` varchar(100) COLLATE utf8mb4_unicode_ci NOT NULL,
  `teams_json` json NOT NULL,
  `toss_winner` varchar(100) COLLATE utf8mb4_unicode_ci NOT NULL,
  `toss_decision` varchar(20) COLLATE utf8mb4_unicode_ci NOT NULL,
  `winner` varchar(100) C

In [294]:
from sqlalchemy.schema import CreateTable

table_name = "batting_player_match_regular"
table = proposed_tables[table_name]

conn = engine.raw_connection()

try:
    cursor = conn.cursor()

    try:
        cursor.execute(
            str(CreateTable(table).compile(dialect=engine.dialect))
        )
        print("TABLE CREATED:", table_name)

    except Exception as exc:
        print("EXACT MYSQL ERROR:")
        print(repr(exc))
        print("\nMYSQL WARNINGS:")

        cursor.execute("SHOW WARNINGS")

        for warning in cursor.fetchall():
            print(warning)

finally:
    conn.close()

EXACT MYSQL ERROR:
OperationalError(1050, "Table 'batting_player_match_regular' already exists")

MYSQL WARNINGS:
('Error', 1050, "Table 'batting_player_match_regular' already exists")


In [295]:
from sqlalchemy import (
    Table, Column, ForeignKeyConstraint,
    PrimaryKeyConstraint, String
)
from sqlalchemy.schema import CreateTable

# Remove previously generated table definitions.
# Preserve the reflected matches table.
for name in csv_data.keys():
    if name in metadata.tables:
        metadata.remove(metadata.tables[name])

proposed_tables = {}

for name, df in csv_data.items():

    columns = []

    for column in df.columns:

        nullable = bool(df[column].isna().any())

        columns.append(
            Column(
                column,
                get_sql_type(column),
                nullable=nullable
            )
        )

    table = Table(
        name,
        metadata,
        *columns,
        PrimaryKeyConstraint("match_id", "player"),
        ForeignKeyConstraint(
            ["match_id"],
            ["matches.match_id"],
            name=f"fk_{name}_matches"
        ),
        mysql_engine="InnoDB",
        mysql_charset="utf8mb4",
        mysql_collate="utf8mb4_unicode_ci"
    )

    proposed_tables[name] = table

    print(f"\nDDL PREVIEW: {name}")
    print(CreateTable(table).compile(dialect=engine.dialect))

print("\nUPDATED DDL GENERATION: PASS")


DDL PREVIEW: batting_player_match_regular

CREATE TABLE batting_player_match_regular (
	match_id VARCHAR(20) NOT NULL, 
	player VARCHAR(100) NOT NULL, 
	is_super_over BOOL NOT NULL, 
	batting_team VARCHAR(100) NOT NULL, 
	runs_scored INTEGER NOT NULL, 
	balls_faced INTEGER NOT NULL, 
	fours INTEGER NOT NULL, 
	sixes INTEGER NOT NULL, 
	dismissals INTEGER NOT NULL, 
	retired_hurt_count INTEGER NOT NULL, 
	strike_rate DOUBLE, 
	batting_average DOUBLE, 
	innings_batted INTEGER NOT NULL, 
	not_out_innings INTEGER NOT NULL, 
	season VARCHAR(30) NOT NULL, 
	date DATE NOT NULL, 
	match_type VARCHAR(30) NOT NULL, 
	PRIMARY KEY (match_id, player), 
	CONSTRAINT fk_batting_player_match_regular_matches FOREIGN KEY(match_id) REFERENCES matches (match_id)
)ENGINE=InnoDB CHARSET=utf8mb4 COLLATE utf8mb4_unicode_ci



DDL PREVIEW: bowling_player_match_regular

CREATE TABLE bowling_player_match_regular (
	match_id VARCHAR(20) NOT NULL, 
	player VARCHAR(100) NOT NULL, 
	is_super_over BOOL NOT NULL, 
	ru

In [296]:

for name, table in proposed_tables.items():
    column = table.c.match_id

    print(
        name,
        "| Type:", column.type,
        "| Collation:", column.type.collation
    )

batting_player_match_regular | Type: VARCHAR(20) | Collation: None
bowling_player_match_regular | Type: VARCHAR(20) | Collation: None
batting_next_match_features | Type: VARCHAR(20) | Collation: None
bowling_next_match_features | Type: VARCHAR(20) | Collation: None


In [297]:
from sqlalchemy.schema import CreateTable

table = proposed_tables["batting_player_match_regular"]

ddl = str(
    CreateTable(table).compile(dialect=engine.dialect)
)

print("MATCH_ID COLUMN IN GENERATED DDL:")

for line in ddl.splitlines():
    if "match_id" in line.lower():
        print(line)

print("\nTABLE COLLATION:")
print(table.dialect_options["mysql"]["collate"])

print("\nCOLUMN COLLATION:")
print(table.c.match_id.type.collation)

MATCH_ID COLUMN IN GENERATED DDL:
	match_id VARCHAR(20) NOT NULL, 
	PRIMARY KEY (match_id, player), 
	CONSTRAINT fk_batting_player_match_regular_matches FOREIGN KEY(match_id) REFERENCES matches (match_id)

TABLE COLLATION:
utf8mb4_unicode_ci

COLUMN COLLATION:
None


In [298]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            TABLE_NAME,
            COLUMN_NAME,
            COLUMN_TYPE,
            CHARACTER_SET_NAME,
            COLLATION_NAME,
            COLUMN_KEY
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'matches'
          AND COLUMN_NAME = 'match_id'
    """))

    for row in result:
        print(dict(row._mapping))

{'TABLE_NAME': 'matches', 'COLUMN_NAME': 'match_id', 'COLUMN_TYPE': 'varchar(20)', 'CHARACTER_SET_NAME': 'utf8mb4', 'COLLATION_NAME': 'utf8mb4_unicode_ci', 'COLUMN_KEY': 'PRI'}


In [299]:
from sqlalchemy.dialects.mysql import VARCHAR
from sqlalchemy import (
    Table, Column, ForeignKeyConstraint
)
from sqlalchemy.schema import CreateTable

# 1. Check MySQL-specific VARCHAR compilation
test_type = VARCHAR(
    20,
    charset="utf8mb4",
    collation="utf8mb4_unicode_ci"
)

print("MYSQL VARCHAR COMPILATION:")
print(test_type.compile(dialect=engine.dialect))

# 2. Remove any previous temporary test table
if "test_fk_compatibility" in metadata.tables:
    metadata.remove(metadata.tables["test_fk_compatibility"])

# 3. Reuse existing metadata containing matches
test_table = Table(
    "test_fk_compatibility",
    metadata,
    Column("match_id", test_type, nullable=False),
    Column("player", VARCHAR(100), primary_key=True),
    ForeignKeyConstraint(
        ["match_id"],
        ["matches.match_id"]
    ),
    mysql_engine="InnoDB",
    mysql_charset="utf8mb4",
    mysql_collate="utf8mb4_unicode_ci"
)

# 4. Compile only; do not execute
print("\nTEST TABLE DDL:")
print(
    CreateTable(test_table).compile(
        dialect=engine.dialect
    )
)

print("\nTEST DDL COMPILATION: PASS")

MYSQL VARCHAR COMPILATION:
VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci

TEST TABLE DDL:

CREATE TABLE test_fk_compatibility (
	match_id VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL, 
	player VARCHAR(100) NOT NULL, 
	PRIMARY KEY (player), 
	FOREIGN KEY(match_id) REFERENCES matches (match_id)
)ENGINE=InnoDB CHARSET=utf8mb4 COLLATE utf8mb4_unicode_ci



TEST DDL COMPILATION: PASS


In [300]:
from sqlalchemy.dialects.mysql import VARCHAR

def get_sql_type(column):

    if column == "match_id":
        return VARCHAR(
            20,
            charset="utf8mb4",
            collation="utf8mb4_unicode_ci"
        )

    if column == "player":
        return VARCHAR(100)

    if column == "season":
        return VARCHAR(30)

    if column == "match_type":
        return VARCHAR(30)

    if column == "batting_team":
        return VARCHAR(100)

    if column in {"date", "previous_match_date"}:
        return Date()

    if column == "is_super_over":
        return Boolean()

    if (
        column in integer_columns
        or column in integer_target_columns
        or column in integer_history_columns
    ):
        return Integer()

    return Double()

In [301]:
for name, table in proposed_tables.items():

    column = table.c.match_id

    print(f"\nTABLE: {name}")
    print("SQLAlchemy type:", type(column.type))
    print("Type representation:", repr(column.type))
    print("Charset:", getattr(column.type, "charset", None))
    print("Collation:", getattr(column.type, "collation", None))

    ddl = str(
        CreateTable(table).compile(dialect=engine.dialect)
    )

    for line in ddl.splitlines():
        if line.strip().startswith("match_id "):
            print("Generated DDL:", line.strip())


TABLE: batting_player_match_regular
SQLAlchemy type: <class 'sqlalchemy.sql.sqltypes.String'>
Type representation: String(length=20)
Charset: None
Collation: None
Generated DDL: match_id VARCHAR(20) NOT NULL,

TABLE: bowling_player_match_regular
SQLAlchemy type: <class 'sqlalchemy.sql.sqltypes.String'>
Type representation: String(length=20)
Charset: None
Collation: None
Generated DDL: match_id VARCHAR(20) NOT NULL,

TABLE: batting_next_match_features
SQLAlchemy type: <class 'sqlalchemy.sql.sqltypes.String'>
Type representation: String(length=20)
Charset: None
Collation: None
Generated DDL: match_id VARCHAR(20) NOT NULL,

TABLE: bowling_next_match_features
SQLAlchemy type: <class 'sqlalchemy.sql.sqltypes.String'>
Type representation: String(length=20)
Charset: None
Collation: None
Generated DDL: match_id VARCHAR(20) NOT NULL,


In [302]:
print(get_sql_type("match_id"))
print(type(get_sql_type("match_id")))
print(get_sql_type("match_id").charset)
print(get_sql_type("match_id").collation)

VARCHAR(20) COLLATE utf8mb4_unicode_ci
<class 'sqlalchemy.dialects.mysql.types.VARCHAR'>
utf8mb4
utf8mb4_unicode_ci


In [303]:
# Remove old proposed tables from metadata.
# Preserve the reflected matches table.

for name in csv_data.keys():
    if name in metadata.tables:
        metadata.remove(metadata.tables[name])

proposed_tables = {}

for name, df in csv_data.items():

    columns = []

    for column in df.columns:

        columns.append(
            Column(
                column,
                get_sql_type(column),
                nullable=bool(df[column].isna().any())
            )
        )

    table = Table(
        name,
        metadata,
        *columns,
        PrimaryKeyConstraint("match_id", "player"),
        ForeignKeyConstraint(
            ["match_id"],
            ["matches.match_id"],
            name=f"fk_{name}_matches"
        ),
        mysql_engine="InnoDB",
        mysql_charset="utf8mb4",
        mysql_collate="utf8mb4_unicode_ci"
    )

    proposed_tables[name] = table

print("FOUR TABLES REBUILT")

FOUR TABLES REBUILT


In [304]:
from sqlalchemy.schema import CreateTable

for name, table in proposed_tables.items():

    ddl = str(
        CreateTable(table).compile(dialect=engine.dialect)
    )

    match_id_line = next(
        line.strip()
        for line in ddl.splitlines()
        if line.strip().startswith("match_id ")
    )

    print(f"\n{name}:")
    print(match_id_line)

    assert "CHARACTER SET utf8mb4" in match_id_line
    assert "COLLATE utf8mb4_unicode_ci" in match_id_line

print("\nALL FOUR MATCH_ID DEFINITIONS: PASS")


batting_player_match_regular:
match_id VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,

bowling_player_match_regular:
match_id VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,

batting_next_match_features:
match_id VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,

bowling_next_match_features:
match_id VARCHAR(20) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,

ALL FOUR MATCH_ID DEFINITIONS: PASS


In [305]:
from pathlib import Path
import subprocess
import sys

script_path = PROJECT_ROOT / "scripts" / "import_csv_to_mysql.py"

if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

print(f"Script found: {script_path}")

# Execute only when an import is actually required.
# Do not rerun blindly against an already populated database.
# subprocess.run(
#     [sys.executable, str(script_path)],
#     cwd=PROJECT_ROOT,
#     check=True
# )

Script found: d:\Git\Personal\ipl-player-performance-v2\scripts\import_csv_to_mysql.py


In [306]:
from sqlalchemy import inspect

created_tables = []

# Always inspect the current database state
inspector = inspect(engine)
existing_tables = set(inspector.get_table_names())

print("EXISTING TABLES:", sorted(existing_tables))

for name, table in proposed_tables.items():

    if name in existing_tables:
        print(f"SKIPPED: {name} already exists")
        continue

    # Use a fresh connection managed by SQLAlchemy
    with engine.begin() as conn:
        table.create(bind=conn, checkfirst=True)

    created_tables.append(name)
    print(f"CREATED: {name}")

print("\nCREATED TABLES:", created_tables)
print("TABLE CREATION PROCESS: COMPLETED")

EXISTING TABLES: ['batting_next_match_features', 'batting_player_match_regular', 'bowling_next_match_features', 'bowling_player_match_regular', 'deliveries', 'innings', 'matches']
SKIPPED: batting_player_match_regular already exists
SKIPPED: bowling_player_match_regular already exists
SKIPPED: batting_next_match_features already exists
SKIPPED: bowling_next_match_features already exists

CREATED TABLES: []
TABLE CREATION PROCESS: COMPLETED


In [307]:
inspector = inspect(engine)

for name in proposed_tables:

    print(f"\nTABLE: {name}")

    assert inspector.has_table(name)

    columns = inspector.get_columns(name)

    print("Actual column count:", len(columns))
    print("Primary key:", inspector.get_pk_constraint(name))
    print("Foreign keys:", inspector.get_foreign_keys(name))

print("\nMYSQL TABLE VERIFICATION: PASS")


TABLE: batting_player_match_regular
Actual column count: 17
Primary key: {'constrained_columns': ['match_id', 'player'], 'name': None}
Foreign keys: [{'name': 'fk_batting_player_match_regular_matches', 'constrained_columns': ['match_id'], 'referred_schema': None, 'referred_table': 'matches', 'referred_columns': ['match_id'], 'options': {}}]

TABLE: bowling_player_match_regular
Actual column count: 16
Primary key: {'constrained_columns': ['match_id', 'player'], 'name': None}
Foreign keys: [{'name': 'fk_bowling_player_match_regular_matches', 'constrained_columns': ['match_id'], 'referred_schema': None, 'referred_table': 'matches', 'referred_columns': ['match_id'], 'options': {}}]

TABLE: batting_next_match_features
Actual column count: 56
Primary key: {'constrained_columns': ['match_id', 'player'], 'name': None}
Foreign keys: [{'name': 'fk_batting_next_match_features_matches', 'constrained_columns': ['match_id'], 'referred_schema': None, 'referred_table': 'matches', 'referred_columns': 

## 9. CSV Preparation and Pre-insertion Validation

Prepare the four processed CSV datasets for MySQL insertion.

Validate column compatibility, data types, missing values,
date columns, and composite primary-key uniqueness.

No database insertion during this checkpoint.

In [308]:
import pandas as pd
from sqlalchemy import inspect

print("=" * 75)
print("CSV PRE-INSERTION VALIDATION")
print("=" * 75)

prepared_data = {}

inspector = inspect(engine)

for name, df in csv_data.items():

    print(f"\nDATASET: {name}")

    prepared = df.copy()

    # Convert match_id to string
    prepared["match_id"] = (
        prepared["match_id"].astype(str)
    )

    # Convert date columns
    for col in ["date", "previous_match_date"]:
        if col in prepared.columns:
            prepared[col] = pd.to_datetime(
                prepared[col],
                errors="raise"
            ).dt.date

    # Validate primary key
    assert prepared["match_id"].notna().all()
    assert prepared["player"].notna().all()

    duplicate_keys = prepared.duplicated(
        subset=["match_id", "player"]
    ).sum()

    assert duplicate_keys == 0

    # Compare CSV and MySQL column names
    csv_columns = set(prepared.columns)

    mysql_columns = {
        col["name"]
        for col in inspector.get_columns(name)
    }

    missing_columns = mysql_columns - csv_columns
    extra_columns = csv_columns - mysql_columns

    assert not missing_columns, (
        f"Missing CSV columns: {missing_columns}"
    )

    assert not extra_columns, (
        f"Unexpected CSV columns: {extra_columns}"
    )

    prepared_data[name] = prepared

    print("Rows:", len(prepared))
    print("Columns:", len(prepared.columns))
    print("Duplicate primary keys:", duplicate_keys)
    print("Missing CSV columns:", missing_columns)
    print("Extra CSV columns:", extra_columns)

print("\nALL FOUR DATASETS PRE-INSERTION VALIDATION: PASS")

CSV PRE-INSERTION VALIDATION

DATASET: batting_player_match_regular
Rows: 18822
Columns: 17
Duplicate primary keys: 0
Missing CSV columns: set()
Extra CSV columns: set()

DATASET: bowling_player_match_regular
Rows: 14700
Columns: 16
Duplicate primary keys: 0
Missing CSV columns: set()
Extra CSV columns: set()

DATASET: batting_next_match_features
Rows: 18822
Columns: 56
Duplicate primary keys: 0
Missing CSV columns: set()
Extra CSV columns: set()

DATASET: bowling_next_match_features
Rows: 14700
Columns: 43
Duplicate primary keys: 0
Missing CSV columns: set()
Extra CSV columns: set()

ALL FOUR DATASETS PRE-INSERTION VALIDATION: PASS


In [309]:
from sqlalchemy import text
from pathlib import Path
import pandas as pd

table_names = [
    "batting_player_match_regular",
    "bowling_player_match_regular",
    "batting_next_match_features",
    "bowling_next_match_features"
]

print("=" * 75)
print("MYSQL vs SOURCE CSV ROW-COUNT DIAGNOSTIC")
print("=" * 75)

with engine.connect() as conn:

    for table_name in table_names:

        db_count = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar_one()

        print(f"\nTable: {table_name}")
        print(f"MySQL rows: {db_count:,}")

        # Compare with the original CSV loaded in Notebook 08
        if table_name in csv_data:
            csv_count = len(csv_data[table_name])
            print(f"CSV rows:   {csv_count:,}")
            print(f"Difference: {db_count - csv_count:,}")

        else:
            print("CSV source not found in csv_data dictionary")

print("\nDIAGNOSTIC COMPLETED — NO DATABASE CHANGES")

MYSQL vs SOURCE CSV ROW-COUNT DIAGNOSTIC

Table: batting_player_match_regular
MySQL rows: 18,822
CSV rows:   18,822
Difference: 0

Table: bowling_player_match_regular
MySQL rows: 14,700
CSV rows:   14,700
Difference: 0

Table: batting_next_match_features
MySQL rows: 18,822
CSV rows:   18,822
Difference: 0

Table: bowling_next_match_features
MySQL rows: 14,700
CSV rows:   14,700
Difference: 0

DIAGNOSTIC COMPLETED — NO DATABASE CHANGES


In [310]:
from sqlalchemy import select, func

print("=" * 75)
print("MYSQL ROW-COUNT RECONCILIATION")
print("=" * 75)

total_rows = 0

with engine.connect() as conn:

    for name, df in prepared_data.items():

        actual = conn.execute(
            select(func.count()).select_from(proposed_tables[name])
        ).scalar_one()

        expected = len(df)

        print(
            f"{name}: Expected={expected:,}, "
            f"Actual={actual:,}"
        )

        assert actual == expected

        total_rows += actual

print("\nTotal database rows:", f"{total_rows:,}")

assert total_rows == 67044

print("\nROW-COUNT RECONCILIATION: PASS")

MYSQL ROW-COUNT RECONCILIATION
batting_player_match_regular: Expected=18,822, Actual=18,822
bowling_player_match_regular: Expected=14,700, Actual=14,700
batting_next_match_features: Expected=18,822, Actual=18,822
bowling_next_match_features: Expected=14,700, Actual=14,700

Total database rows: 67,044

ROW-COUNT RECONCILIATION: PASS


In [311]:
from sqlalchemy import text

print("=" * 75)
print("MYSQL PRIMARY KEY AND FOREIGN KEY VALIDATION")
print("=" * 75)

with engine.connect() as conn:

    for name in proposed_tables:

        table = proposed_tables[name]

        # Duplicate primary-key check
        duplicate_count = conn.execute(text(f"""
            SELECT COUNT(*)
            FROM (
                SELECT match_id, player
                FROM `{name}`
                GROUP BY match_id, player
                HAVING COUNT(*) > 1
            ) AS duplicates
        """)).scalar_one()

        # Foreign-key orphan check
        orphan_count = conn.execute(text(f"""
            SELECT COUNT(*)
            FROM `{name}` child
            LEFT JOIN matches parent
                ON child.match_id = parent.match_id
            WHERE parent.match_id IS NULL
        """)).scalar_one()

        print(f"\nTABLE: {name}")
        print("Duplicate PK groups:", duplicate_count)
        print("Orphan match IDs:", orphan_count)

        assert duplicate_count == 0
        assert orphan_count == 0

print("\nPRIMARY KEY AND FOREIGN KEY VALIDATION: PASS")

MYSQL PRIMARY KEY AND FOREIGN KEY VALIDATION

TABLE: batting_player_match_regular
Duplicate PK groups: 0
Orphan match IDs: 0

TABLE: bowling_player_match_regular
Duplicate PK groups: 0
Orphan match IDs: 0

TABLE: batting_next_match_features
Duplicate PK groups: 0
Orphan match IDs: 0

TABLE: bowling_next_match_features
Duplicate PK groups: 0
Orphan match IDs: 0

PRIMARY KEY AND FOREIGN KEY VALIDATION: PASS


In [312]:
from sqlalchemy import text
import pandas as pd

print("=" * 75)
print("MYSQL NULL VALUE RECONCILIATION")
print("=" * 75)

null_validation = []

with engine.connect() as conn:

    for name, source_df in prepared_data.items():

        print(f"\nTABLE: {name}")

        for column in source_df.columns:

            expected_nulls = int(source_df[column].isna().sum())

            actual_nulls = conn.execute(
                text(f"""
                    SELECT COUNT(*)
                    FROM `{name}`
                    WHERE `{column}` IS NULL
                """)
            ).scalar_one()

            status = (
                "PASS"
                if expected_nulls == actual_nulls
                else "FAIL"
            )

            if expected_nulls > 0 or actual_nulls > 0:
                print(
                    f"{column}: CSV={expected_nulls}, "
                    f"MySQL={actual_nulls} [{status}]"
                )

            null_validation.append({
                "table": name,
                "column": column,
                "csv_nulls": expected_nulls,
                "mysql_nulls": actual_nulls,
                "status": status
            })

null_audit = pd.DataFrame(null_validation)

print("\nNULL VALIDATION SUMMARY")
print(null_audit["status"].value_counts())

assert (null_audit["status"] == "PASS").all()

print("\nNULL VALUE RECONCILIATION: PASS")

MYSQL NULL VALUE RECONCILIATION

TABLE: batting_player_match_regular
strike_rate: CSV=59, MySQL=59 [PASS]
batting_average: CSV=4167, MySQL=4167 [PASS]

TABLE: bowling_player_match_regular
economy_rate: CSV=1, MySQL=1 [PASS]
bowling_average: CSV=6251, MySQL=6251 [PASS]
bowling_strike_rate: CSV=6251, MySQL=6251 [PASS]

TABLE: batting_next_match_features


strike_rate: CSV=59, MySQL=59 [PASS]
batting_average: CSV=4167, MySQL=4167 [PASS]
target_next_match_runs_scored: CSV=739, MySQL=739 [PASS]
target_next_match_balls_faced: CSV=739, MySQL=739 [PASS]
target_next_match_fours: CSV=739, MySQL=739 [PASS]
target_next_match_sixes: CSV=739, MySQL=739 [PASS]
target_next_match_strike_rate: CSV=794, MySQL=794 [PASS]
target_next_match_dismissals: CSV=739, MySQL=739 [PASS]
previous_match_date: CSV=739, MySQL=739 [PASS]
days_since_previous_match: CSV=739, MySQL=739 [PASS]
previous_runs_scored: CSV=739, MySQL=739 [PASS]
runs_scored_rolling_mean_3: CSV=739, MySQL=739 [PASS]
runs_scored_rolling_mean_5: CSV=739, MySQL=739 [PASS]
runs_scored_rolling_mean_10: CSV=739, MySQL=739 [PASS]
runs_scored_expanding_mean: CSV=739, MySQL=739 [PASS]
previous_balls_faced: CSV=739, MySQL=739 [PASS]
balls_faced_rolling_mean_3: CSV=739, MySQL=739 [PASS]
balls_faced_rolling_mean_5: CSV=739, MySQL=739 [PASS]
balls_faced_rolling_mean_10: CSV=739, MySQL=739 [PASS]
balls_faced_e

In [313]:
from sqlalchemy import text
import numpy as np

print("=" * 75)
print("NUMERIC AGGREGATE RECONCILIATION")
print("=" * 75)

aggregate_columns = {
    "batting_player_match_regular": [
        "runs_scored",
        "balls_faced",
        "fours",
        "sixes",
        "dismissals"
    ],
    "bowling_player_match_regular": [
        "runs_conceded",
        "legal_balls",
        "wickets"
    ],
    "batting_next_match_features": [
        "target_next_match_runs_scored",
        "target_next_match_balls_faced",
        "target_next_match_fours",
        "target_next_match_sixes",
        "matches_played_before"
    ],
    "bowling_next_match_features": [
        "target_next_match_wickets",
        "target_next_match_runs_conceded",
        "target_next_match_legal_balls",
        "matches_played_before"
    ]
}

aggregate_results = []

with engine.connect() as conn:

    for name, columns in aggregate_columns.items():

        source_df = prepared_data[name]

        print(f"\nTABLE: {name}")

        for column in columns:

            expected = source_df[column].sum()

            actual = conn.execute(
                text(f"""
                    SELECT SUM(`{column}`)
                    FROM `{name}`
                """)
            ).scalar_one()

            difference = abs(float(expected) - float(actual))

            status = "PASS" if np.isclose(
                float(expected),
                float(actual),
                rtol=1e-9,
                atol=1e-9
            ) else "FAIL"

            print(
                f"{column}: CSV={expected}, "
                f"MySQL={actual}, Difference={difference}, "
                f"Status={status}"
            )

            aggregate_results.append(status)

assert all(x == "PASS" for x in aggregate_results)

print("\nNUMERIC AGGREGATE RECONCILIATION: PASS")

NUMERIC AGGREGATE RECONCILIATION

TABLE: batting_player_match_regular
runs_scored: CSV=381235, MySQL=381235, Difference=0.0, Status=PASS
balls_faced: CSV=285686, MySQL=285686, Difference=0.0, Status=PASS
fours: CSV=34420, MySQL=34420, Difference=0.0, Status=PASS
sixes: CSV=15763, MySQL=15763, Difference=0.0, Status=PASS
dismissals: CSV=14655, MySQL=14655, Difference=0.0, Status=PASS

TABLE: bowling_player_match_regular
runs_conceded: CSV=394281, MySQL=394281, Difference=0.0, Status=PASS
legal_balls: CSV=284465, MySQL=284465, Difference=0.0, Status=PASS
wickets: CSV=13460, MySQL=13460, Difference=0.0, Status=PASS

TABLE: batting_next_match_features
target_next_match_runs_scored: CSV=373004.0, MySQL=373004, Difference=0.0, Status=PASS
target_next_match_balls_faced: CSV=278729.0, MySQL=278729, Difference=0.0, Status=PASS
target_next_match_fours: CSV=33667.0, MySQL=33667, Difference=0.0, Status=PASS
target_next_match_sixes: CSV=15454.0, MySQL=15454, Difference=0.0, Status=PASS
matches_play

In [314]:

from sqlalchemy import text

expected_original_counts = {
    "matches": 1243,
    "innings": 2514,
    "deliveries": 295732
}

print("=" * 75)
print("ORIGINAL DATABASE PRESERVATION AUDIT")
print("=" * 75)

with engine.connect() as conn:

    for table_name, expected in expected_original_counts.items():

        actual = conn.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar_one()

        status = "PASS" if actual == expected else "FAIL"

        print(
            f"{table_name}: "
            f"Expected={expected:,}, "
            f"Actual={actual:,}, "
            f"Status={status}"
        )

        assert actual == expected

print("\nORIGINAL DATABASE PRESERVATION: PASS")

ORIGINAL DATABASE PRESERVATION AUDIT
matches: Expected=1,243, Actual=1,243, Status=PASS
innings: Expected=2,514, Actual=2,514, Status=PASS
deliveries: Expected=295,732, Actual=295,732, Status=PASS

ORIGINAL DATABASE PRESERVATION: PASS
